In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("PyTorch version:", torch.__version__)
print("Sab ready hai!")

PyTorch version: 2.13.0+cpu
Sab ready hai!


In [2]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import numpy as np

# ==========================================
# STEP 1: REAL DATA SETUP & SCALING
# ==========================================
# 3 Customers ka Raw Data (3 Columns: Order Value, Delivery Days, Age)
raw_X = np.array([
    [150.0, 2.0, 25.0],   # Customer 1: $150, 2 days delivery, 25 yrs old
    [30.0,  10.0, 60.0],  # Customer 2: $30, 10 days delivery, 60 yrs old
    [200.0, 1.0, 30.0]    # Customer 3: $200, 1 day delivery, 30 yrs old
])

# Actual Outcomes (1 = Order Completed, 0 = Order Cancelled)
raw_y = np.array([
    [1.0], 
    [0.0], 
    [1.0]
])

# Data Ko Scale Karein (Teeno columns ko ek hi balance range mein laane ke liye)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(raw_X)

# PyTorch Tensors Mein Convert Karein (2D Matrix Format)
X_train = torch.FloatTensor(X_scaled)
y_train = torch.FloatTensor(raw_y)

print("Step 1 Done: Scaled Tensors Ready!")
print("X_train Shape:", X_train.shape) # (3 rows, 3 columns)

# ==========================================
# STEP 2: MODEL ARCHITECTURE
# ==========================================
class OrderPredictor(nn.Module):
    def __init__(self, input_features):
        super(OrderPredictor, self).__init__()
        # Assembly: Layers Fit Kar Rahe Hain
        self.layer1 = nn.Linear(input_features, 8) # 3 Input Sockets -> 8 Neurons
        self.layer2 = nn.Linear(8, 1)              # 8 Neurons -> 1 Output Score
        
        self.relu = nn.ReLU()       # Negative scores ko 0 karne ke liye
        self.sigmoid = nn.Sigmoid() # Answer ko 0.0 se 1.0 (% chance) mein laane ke liye

    def forward(self, x):
        # Execution: Actual Data 'x' Yahan Se Guzarta Hai
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.sigmoid(x)
        return x

# Model Banayein (3 inputs ke sath)
model = OrderPredictor(input_features=3)
print("\nStep 2 Done: Model Architecture Ready!")

# ==========================================
# STEP 3: RULES (LOSS FUNCTION & OPTIMIZER)
# ==========================================
# Galti napne ka paimana (Binary Classification ke liye)
criterion = nn.BCELoss()

# Model ke weights sahi karne wala engineer (Adam Optimizer)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Step 3 Done: Loss & Optimizer Set!")

# ==========================================
# STEP 4: TRAINING LOOP (SEEKHNE KA PROCESS)
# ==========================================
epochs = 200 # Model 200 baar galti karke seekhega

print("\nStep 4: Training Started...")
for epoch in range(1, epochs + 1):
    # A. GUESS: Model Prediction
    predictions = model(X_train)
    
    # B. GALTI: Real answer se kitna farq hai?
    loss = criterion(predictions, y_train)
    
    # C. FIX: Weights ko update karna
    optimizer.zero_grad() # Purani mistake history saaf
    loss.backward()        # Galti ka hisaab (Backpropagation)
    optimizer.step()       # Weights ko sahi taraf adjust karna
    
    # Har 50 epochs baad loss print karein
    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{epochs} - Loss (Galti): {loss.item():.4f}")

# ==========================================
# STEP 5: PREDICT ON A NEW CUSTOMER
# ==========================================
model.eval() # Testing mode ON

# Naya Customer: $180 order value, 2 days delivery, 28 yrs old
new_customer_raw = np.array([[180.0, 2.0, 28.0]])

# Pehle purane scaler se hi scale karein
new_customer_scaled = scaler.transform(new_customer_raw)
new_customer_tensor = torch.FloatTensor(new_customer_scaled)

with torch.no_grad():
    completion_chance = model(new_customer_tensor).item()

print("\n================ PREDICTION RESULT ================")
print(f"Naye Customer ka Order Complete Hone Ka Chance: {completion_chance * 100:.2f}%")

Step 1 Done: Scaled Tensors Ready!
X_train Shape: torch.Size([3, 3])

Step 2 Done: Model Architecture Ready!
Step 3 Done: Loss & Optimizer Set!

Step 4: Training Started...
Epoch 50/200 - Loss (Galti): 0.0613
Epoch 100/200 - Loss (Galti): 0.0101
Epoch 150/200 - Loss (Galti): 0.0046
Epoch 200/200 - Loss (Galti): 0.0027

================ PREDICTION RESULT ================
Naye Customer ka Order Complete Hone Ka Chance: 99.69%
